# TABR datatest event-level — accuracy improvement experiments

Notebook pembuktian 7 strategi naik accuracy:
1. ranking by accuracy
2. fine threshold search
3. weighted ensemble
4. fallback rule
5. threshold per question
6. threshold per date
7. event-noise filtering


In [1]:
from __future__ import annotations

import gc
import json
import os
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import Markdown, display
from scipy.special import expit
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support

sns.set_theme(style='whitegrid')
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

TABR_ROOT = ROOT / 'third_party' / 'tabular-dl-tabr-official'
os.environ['PROJECT_DIR'] = str(TABR_ROOT)
if str(TABR_ROOT) not in sys.path:
    sys.path.append(str(TABR_ROOT))

import lib
from bin.tabr import Model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EXP_ROOT = TABR_ROOT / 'exp' / 'tabr'
DATA_ROOT = TABR_ROOT / 'data'
MODEL_SPECS = [
    {
        'label': 'qholdout_q123_q4_q5',
        'exp_name': 'convat_apex_anxiety_qholdout_q123_q4_q5',
        'mode': 'qholdout',
        'aggregation': 'mean',
        'threshold': 0.225,
    },
    {
        'label': 'qwalk_q12_q3_q4',
        'exp_name': 'convat_apex_anxiety_qwalk_q12_q3_q4',
        'mode': 'qwalkforward',
        'aggregation': 'mean',
        'threshold': 0.200,
    },
    {
        'label': 'qwalk_q123_q4_q5',
        'exp_name': 'convat_apex_anxiety_qwalk_q123_q4_q5',
        'mode': 'qwalkforward',
        'aggregation': 'mean',
        'threshold': 0.300,
    },
]
TOP_K = 3
EVENT_METHODS = ['mean', 'top3_mean', 'p90', 'max']
THRESHOLDS = np.round(np.arange(0.05, 0.951, 0.025), 3)
THRESHOLDS_FINE = np.round(np.arange(0.10, 0.251, 0.005), 3)
WEIGHT_GRID = np.round(np.arange(0.0, 1.01, 0.1), 2)
PRIMARY_METRIC = 'macro_f1'
FPS = 25
EVAL_ROOT = ROOT / 'output/apex/datatest/event_level_accuracy_experiments'
CM_ROOT = EVAL_ROOT / 'confusion_matrices'
EVAL_ROOT.mkdir(parents=True, exist_ok=True)
CM_ROOT.mkdir(parents=True, exist_ok=True)
display(pd.DataFrame(MODEL_SPECS))
print(f'Root: {ROOT}')
print(f'Device: {device}')


,label,exp_name,mode,aggregation,threshold
0,qholdout_q123_q4_q5,convat_apex_anxiety_qholdout_q123_q4_q5,qholdout,mean,0.225
1,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,qwalkforward,mean,0.200
2,qwalk_q123_q4_q5,convat_apex_anxiety_qwalk_q123_q4_q5,qwalkforward,mean,0.300


Root: /home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st
Device: cuda


In [2]:
FEATURE_PATHS = [
    ROOT / 'output/apex/datatest/features/15_04_2026/poc_abs_test.xlsx',
    ROOT / 'output/apex/datatest/features/17_04_2026/poc_abs_test.xlsx',
]

SELECTED_HIGH = [
    'naufal_abid_aurizky',
    'hafif_nurrahmad',
    'bisma_adhiaksa',
    'tsabita_keizya_vanya_putri',
    'raihan_daffa_izzuddin',
    'andy',
    'dicky_darmawan',
    'fata_haidar_aly',
    'muhammad_ihsan_rahmatullah',
    'raymon_devtant',
]

SELECTED_LOW = [
    'muhammad_daniel_assaqovi',
    'sileysa_faedatul_nuraini',
    'ilham_dharma_atmaja',
    'abelas_solihin',
    'dewi_anggraini_meita_sari',
    'farrel_andika_chandra',
    'muhammad_nuril_huda',
    'muhammad_rizky_firdiansyah',
    'rachmad_aprisandhy',
    'zukhrufian_abdullah',
]

SELECTED_SUBJECTS = SELECTED_HIGH + SELECTED_LOW
EXPECTED_LABEL_BY_BASE = {
    **{name: 'anxiety_tinggi' for name in SELECTED_HIGH},
    **{name: 'anxiety_rendah' for name in SELECTED_LOW},
}

def label_to_int(label: str) -> int:
    return 1 if str(label) in {'anxiety_tinggi', 'anxiety', '1', 'tinggi'} else 0


def int_to_label(value: int) -> str:
    return 'anxiety_tinggi' if int(value) == 1 else 'anxiety_rendah'


def participant_base(value: str) -> str:
    return re.sub(r'_\d+$', '', str(value))


def filter_selected_subjects(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['participant_base'] = out['participant_raw'].map(participant_base)
    out = out[out['participant_base'].isin(SELECTED_SUBJECTS)].copy()
    out = out[out['label'] == out['participant_base'].map(EXPECTED_LABEL_BY_BASE)].copy()
    return out


def load_datatest_features(paths: list[Path]) -> pd.DataFrame:
    dfs = []
    for path in paths:
        if not path.exists():
            raise FileNotFoundError(f'Feature file not found: {path}')
        df = pd.read_excel(path)
        if 'date' not in df.columns:
            df['date'] = path.parent.name
        dfs.append(df)
    out = pd.concat(dfs, ignore_index=True)
    out = filter_selected_subjects(out)
    out['true_label'] = out['label'].map(label_to_int).astype(int)
    out['event_key'] = (
        out['date'].astype(str) + '|' + out['participant_raw'].astype(str) + '|' + out['question'].astype(str) + '|' +
        out['sample'].astype(str) + '|' + out['event_clip'].astype(str) + '|' + out['event_no'].astype(str)
    )
    out['kode_event'] = out['sample'].astype(str) + ' / ' + out['event_clip'].astype(str)
    return out


frame_df = load_datatest_features(FEATURE_PATHS)
missing = sorted(set(SELECTED_SUBJECTS) - set(frame_df['participant_base'].unique()))
if missing:
    raise ValueError(f'Missing selected subjects: {missing}')

subject_label_counts = (
    frame_df.drop_duplicates('participant_base')['label']
    .value_counts()
    .reindex(['anxiety_tinggi', 'anxiety_rendah'], fill_value=0)
)
if subject_label_counts.to_dict() != {'anxiety_tinggi': 10, 'anxiety_rendah': 10}:
    raise ValueError(f'Expected 10 tinggi + 10 rendah, got: {subject_label_counts.to_dict()}')
print('Jumlah subjek:', subject_label_counts.to_dict())

datatest_table = (
    frame_df
    .groupby('event_key', sort=False)
    .agg(
        kode_video=('kode_event', 'first'),
        kondisi=('condition', 'first'),
        label=('label', 'first'),
        jumlah_frame=('frame', 'count'),
    )
    .reset_index(drop=True)
)
datatest_table.insert(0, 'No', np.arange(1, len(datatest_table) + 1))
datatest_table['durasi'] = datatest_table['jumlah_frame'] / FPS
datatest_table = datatest_table[['No', 'kode_video', 'kondisi', 'label', 'durasi', 'jumlah_frame']]
datatest_table = datatest_table.rename(columns={
    'kode_video': 'Kode Video',
    'kondisi': 'Kondisi',
    'label': 'Label',
    'durasi': 'Durasi',
    'jumlah_frame': 'Jumlah Frame',
})
display(Markdown('## Tabel data test event-level'))
display(datatest_table)


Jumlah subjek: {'anxiety_tinggi': 10, 'anxiety_rendah': 10}


## Tabel data test event-level

,No,Kode Video,Kondisi,Label,Durasi,Jumlah Frame
0,1,Abelas Solihin_1_15d591ce-051a-47f2-ac38-367c1...,anxiety_rendah,anxiety_rendah,0.28,7
1,2,Abelas Solihin_1_15d591ce-051a-47f2-ac38-367c1...,anxiety_rendah,anxiety_rendah,0.64,16
2,3,Abelas Solihin_1_15d591ce-051a-47f2-ac38-367c1...,anxiety_rendah,anxiety_rendah,0.96,24
3,4,Abelas Solihin_1_15d591ce-051a-47f2-ac38-367c1...,anxiety_rendah,anxiety_rendah,1.24,31
4,5,Abelas Solihin_1_15d591ce-051a-47f2-ac38-367c1...,anxiety_rendah,anxiety_rendah,1.72,43
...,...,...,...,...,...,...
687,688,Raihan Daffa Izzuddin_5_e1e6b3e5-a2d1-4832-a1f...,anxiety_tinggi,anxiety_tinggi,0.40,10
688,689,Raihan Daffa Izzuddin_5_e1e6b3e5-a2d1-4832-a1f...,anxiety_tinggi,anxiety_tinggi,0.16,4
689,690,Raihan Daffa Izzuddin_5_e1e6b3e5-a2d1-4832-a1f...,anxiety_tinggi,anxiety_tinggi,0.32,8
690,691,Raihan Daffa Izzuddin_5_e1e6b3e5-a2d1-4832-a1f...,anxiety_tinggi,anxiety_tinggi,0.20,5


In [3]:
def load_best_seed_report(exp_name: str) -> tuple[int, dict, Path]:
    eval_dir = EXP_ROOT / exp_name / '0-evaluation'
    rows = []
    for run_dir in sorted([x for x in eval_dir.iterdir() if x.is_dir() and x.name.isdigit()]):
        report_path = run_dir / 'report.json'
        if not report_path.exists():
            continue
        report = json.loads(report_path.read_text())
        rows.append((int(run_dir.name), report, run_dir))
    if not rows:
        raise FileNotFoundError(f'No report.json found for {exp_name}')
    return max(rows, key=lambda x: x[1]['metrics']['val']['score'])


def load_run_config(exp_name: str, best_seed: int, report: dict) -> dict:
    seed_toml = EXP_ROOT / exp_name / f'{best_seed}.toml'
    if seed_toml.exists():
        return lib.load_config(seed_toml)
    if 'config' in report:
        return report['config']
    tuning_toml = EXP_ROOT / exp_name / '0-tuning.toml'
    if tuning_toml.exists():
        return lib.load_config(tuning_toml)
    raise FileNotFoundError(f'No config found for {exp_name} seed {best_seed}')


def resolve_data_dir(data_path: str) -> Path:
    if data_path.startswith(':data/'):
        return DATA_ROOT / data_path.removeprefix(':data/')
    return Path(data_path)


def load_model_bundle(exp_name: str) -> dict:
    best_seed, report, best_run_dir = load_best_seed_report(exp_name)
    cfg = load_run_config(exp_name, best_seed, report)
    data_dir = resolve_data_dir(cfg['data']['path'])
    feature_cols = json.loads((data_dir / 'feature_cols.json').read_text())
    dataset = lib.build_dataset(**cfg['data']).to_torch(device)
    model = Model(
        n_num_features=dataset.n_num_features,
        n_bin_features=dataset.n_bin_features,
        cat_cardinalities=dataset.cat_cardinalities(),
        n_classes=dataset.n_classes(),
        **cfg['model'],
    ).to(device)
    checkpoint = lib.load_checkpoint(best_run_dir)
    model.load_state_dict(checkpoint['model'])
    model.eval()
    candidate_x = {key[2:]: dataset.data[key]['train'] for key in dataset.data if key.startswith('X_')}
    candidate_y = dataset.Y['train']
    return {
        'model': model,
        'dataset': dataset,
        'candidate_x': candidate_x,
        'candidate_y': candidate_y,
        'feature_cols': feature_cols,
        'context_size': cfg['context_size'],
        'best_seed': best_seed,
        'data_path': cfg['data']['path'],
    }


def release_model_bundle(bundle: dict) -> None:
    del bundle
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [4]:
@torch.inference_mode()
def predict_frame_probs(bundle: dict, df: pd.DataFrame, batch_size: int = 512, progress_every: int = 500) -> np.ndarray:
    feature_cols = bundle['feature_cols']
    missing = [c for c in feature_cols if c not in df.columns]
    if missing:
        raise KeyError(f'Missing feature cols: {missing[:20]}')
    x_np = df[feature_cols].to_numpy(dtype=np.float32)
    probs = []
    total = len(x_np)
    for start in range(0, total, batch_size):
        if progress_every and start % progress_every < batch_size:
            print(f'  frame {start}/{total}')
        xb = torch.as_tensor(x_np[start:start + batch_size], device=device)
        for i in range(xb.shape[0]):
            x = {'num': xb[i:i+1]}
            logit = bundle['model'](
                x_=x,
                y=None,
                candidate_x_=bundle['candidate_x'],
                candidate_y=bundle['candidate_y'],
                context_size=bundle['context_size'],
                is_train=False,
            ).squeeze(-1).item()
            probs.append(float(expit(logit)))
    return np.asarray(probs, dtype=float)


def aggregate_probs(values, method: str, top_k: int = TOP_K) -> float:
    arr = np.asarray(list(values), dtype=float)
    if len(arr) == 0:
        return np.nan
    if method == 'mean':
        return float(np.mean(arr))
    if method == 'max':
        return float(np.max(arr))
    if method == 'p90':
        return float(np.quantile(arr, 0.90))
    if method == 'top3_mean':
        k = min(top_k, len(arr))
        return float(np.mean(np.sort(arr)[-k:]))
    raise ValueError(f'Unknown aggregation method: {method}')


In [5]:
def make_event_predictions(frame_df: pd.DataFrame, spec: dict, method: str, threshold: float) -> pd.DataFrame:
    rows = []
    meta_cols = ['event_key', 'kode_event', 'condition', 'label', 'true_label', 'participant_base', 'participant_raw', 'question', 'sample', 'event_clip', 'event_no']
    for event_key, g in frame_df.groupby('event_key', sort=False):
        first = g.iloc[0]
        prob = aggregate_probs(g['prob_anxiety_tinggi'], method)
        row = {col: first[col] for col in meta_cols if col in g.columns}
        row.update({
            'model_label': spec['label'],
            'exp_name': spec['exp_name'],
            'aggregation': method,
            'threshold': float(threshold),
            'agg_prob_anxiety_tinggi': prob,
            'pred_label': int(prob >= threshold),
            'n_frames': len(g),
        })
        rows.append(row)
    return pd.DataFrame(rows)


def metrics_dict(df: pd.DataFrame, spec: dict | None, method: str, threshold: float, name: str | None = None) -> dict:
    y_true = df['true_label'].astype(int).to_numpy()
    y_pred = df['pred_label'].astype(int).to_numpy()
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return {
        'model': name or spec['label'],
        'exp_name': '' if spec is None else spec['exp_name'],
        'aggregation': method,
        'threshold': float(threshold),
        'n': len(df),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision,
        'recall': recall,
        'f1-score': f1,
        'macro_f1': f1,
    }


def evaluate_grid(frame_df: pd.DataFrame, spec: dict) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    rows = []
    predictions = {}
    for method in EVENT_METHODS:
        base = make_event_predictions(frame_df, spec, method, threshold=0.5)
        probs = base['agg_prob_anxiety_tinggi'].to_numpy()
        for threshold in THRESHOLDS:
            pred_df = base.copy()
            pred_df['threshold'] = float(threshold)
            pred_df['pred_label'] = (probs >= float(threshold)).astype(int)
            row = metrics_dict(pred_df, spec, method, float(threshold))
            rows.append(row)
            predictions[(method, float(threshold))] = pred_df
    return pd.DataFrame(rows), predictions


def choose_best(summary_df: pd.DataFrame) -> pd.Series:
    return (
        summary_df
        .sort_values(['macro_f1', 'recall', 'accuracy'], ascending=False)
        .iloc[0]
    )


def make_prediction_table(prediction_df: pd.DataFrame) -> pd.DataFrame:
    table = prediction_df[['kode_event', 'true_label', 'pred_label']].copy()
    table.insert(0, 'No', np.arange(1, len(table) + 1))
    table['Label Aktual'] = table['true_label'].map(int_to_label)
    table['Label Prediksi'] = table['pred_label'].map(int_to_label)
    table['Status'] = np.where(table['true_label'] == table['pred_label'], 'Benar', 'Salah')
    table = table.rename(columns={'kode_event': 'Kode Video'})
    return table[['No', 'Kode Video', 'Label Aktual', 'Label Prediksi', 'Status']]


def safe_name(value: str) -> str:
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value))


def plot_confusion(df: pd.DataFrame, title: str, filename: str) -> Path:
    y_true = df['true_label'].astype(int).to_numpy()
    y_pred = df['pred_label'].astype(int).to_numpy()
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    path = CM_ROOT / filename
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', cbar=False,
        xticklabels=['rendah', 'tinggi'], yticklabels=['rendah', 'tinggi'], ax=ax,
    )
    ax.set_xlabel('Label Prediksi')
    ax.set_ylabel('Label Aktual')
    ax.set_title(title)
    plt.tight_layout()
    fig.savefig(path, dpi=160)
    plt.show()
    plt.close(fig)
    return path


In [6]:
all_frame_predictions = []

for spec in MODEL_SPECS:
    display(Markdown(f"# Load predictions: `{spec['label']}`"))
    bundle = load_model_bundle(spec['exp_name'])
    print('best_seed=', bundle['best_seed'], 'features=', len(bundle['feature_cols']), 'data_path=', bundle['data_path'])
    try:
        work = frame_df.copy()
        work['prob_anxiety_tinggi'] = predict_frame_probs(bundle, work)
        work['model_label'] = spec['label']
        work['exp_name'] = spec['exp_name']
        all_frame_predictions.append(work)
    finally:
        release_model_bundle(bundle)

frame_predictions_df = pd.concat(all_frame_predictions, ignore_index=True)
display(frame_predictions_df.groupby('model_label').size().reset_index(name='frames'))


# Load predictions: `qholdout_q123_q4_q5`

2026-05-19 15:56:42.542 | INFO     | lib.data:build_dataset:338 - Using cached dataset: build_dataset__convat_apex_anxiety_qholdout_q123_q4_q5__None__None__None__None__72__d10c1508d8934c1de65e6394f780722e.pickle


best_seed= 13 features= 392 data_path= :data/convat_apex_anxiety_qholdout_q123_q4_q5
  frame 0/8223
  frame 512/8223
  frame 1024/8223
  frame 1536/8223
  frame 2048/8223
  frame 2560/8223
  frame 3072/8223
  frame 3584/8223
  frame 4096/8223
  frame 4608/8223
  frame 5120/8223
  frame 5632/8223
  frame 6144/8223
  frame 6656/8223
  frame 7168/8223
  frame 7680/8223
  frame 8192/8223


# Load predictions: `qwalk_q12_q3_q4`

2026-05-19 15:57:35.128 | INFO     | lib.data:build_dataset:338 - Using cached dataset: build_dataset__convat_apex_anxiety_qwalk_q12_q3_q4__None__None__None__None__72__d10c1508d8934c1de65e6394f780722e.pickle


best_seed= 4 features= 392 data_path= :data/convat_apex_anxiety_qwalk_q12_q3_q4
  frame 0/8223
  frame 512/8223
  frame 1024/8223
  frame 1536/8223
  frame 2048/8223
  frame 2560/8223
  frame 3072/8223
  frame 3584/8223
  frame 4096/8223
  frame 4608/8223
  frame 5120/8223
  frame 5632/8223
  frame 6144/8223
  frame 6656/8223
  frame 7168/8223
  frame 7680/8223
  frame 8192/8223


# Load predictions: `qwalk_q123_q4_q5`

2026-05-19 15:58:12.419 | INFO     | lib.data:build_dataset:338 - Using cached dataset: build_dataset__convat_apex_anxiety_qwalk_q123_q4_q5__None__None__None__None__72__d10c1508d8934c1de65e6394f780722e.pickle


best_seed= 5 features= 392 data_path= :data/convat_apex_anxiety_qwalk_q123_q4_q5
  frame 0/8223
  frame 512/8223
  frame 1024/8223
  frame 1536/8223
  frame 2048/8223
  frame 2560/8223
  frame 3072/8223
  frame 3584/8223
  frame 4096/8223
  frame 4608/8223
  frame 5120/8223
  frame 5632/8223
  frame 6144/8223
  frame 6656/8223
  frame 7168/8223
  frame 7680/8223
  frame 8192/8223


,model_label,frames
0,qholdout_q123_q4_q5,8223
1,qwalk_q123_q4_q5,8223
2,qwalk_q12_q3_q4,8223


In [7]:
def accuracy_ranking_for_model(frame_pred_df: pd.DataFrame, spec: dict, methods: list[str], thresholds: np.ndarray) -> tuple[pd.DataFrame, dict]:
    rows = []
    pred_lookup = {}
    for method in methods:
        base = make_event_predictions(frame_pred_df, spec, method, 0.5)
        probs = base['agg_prob_anxiety_tinggi'].to_numpy()
        for threshold in thresholds:
            pred = base.copy()
            pred['threshold'] = float(threshold)
            pred['pred_label'] = (probs >= float(threshold)).astype(int)
            row = metrics_dict(pred, spec, method, float(threshold))
            rows.append(row)
            pred_lookup[(spec['label'], method, float(threshold))] = pred
    return pd.DataFrame(rows), pred_lookup

def add_balance_cols(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['recall_rendah'] = out['tn'] / (out['tn'] + out['fp'])
    out['recall_tinggi'] = out['tp'] / (out['tp'] + out['fn'])
    out['recall_gap'] = (out['recall_rendah'] - out['recall_tinggi']).abs()
    out['balance_score'] = out['macro_f1'] - out['recall_gap']
    return out

def event_prob_table(frame_pred_df: pd.DataFrame, spec: dict, method: str) -> pd.DataFrame:
    return make_event_predictions(frame_pred_df, spec, method, 0.5).drop(columns=['pred_label', 'threshold'])

def apply_threshold(base: pd.DataFrame, threshold: float, name: str, aggregation: str) -> pd.DataFrame:
    out = base.copy()
    out['pred_label'] = (out['agg_prob_anxiety_tinggi'] >= float(threshold)).astype(int)
    out['threshold'] = float(threshold)
    out['model_label'] = name
    out['aggregation'] = aggregation
    return out

def metrics_for_name(df: pd.DataFrame, name: str, aggregation: str, threshold: float) -> dict:
    return metrics_dict(df, None, aggregation, threshold, name=name)


In [8]:
display(Markdown('# 1–2. Ranking by accuracy + fine threshold search'))

all_acc_rows = []
all_fine_rows = []
fine_lookup = {}

for spec in MODEL_SPECS:
    model_frames = frame_predictions_df[frame_predictions_df['model_label'] == spec['label']].copy()
    coarse_df, _ = accuracy_ranking_for_model(model_frames, spec, EVENT_METHODS, THRESHOLDS)
    fine_df, lookup = accuracy_ranking_for_model(model_frames, spec, EVENT_METHODS, THRESHOLDS_FINE)
    all_acc_rows.append(coarse_df)
    all_fine_rows.append(fine_df)
    fine_lookup.update(lookup)

accuracy_ranking_df = add_balance_cols(pd.concat(all_acc_rows, ignore_index=True))
fine_threshold_df = add_balance_cols(pd.concat(all_fine_rows, ignore_index=True))

display(Markdown('## Coarse ranking by accuracy'))
display(accuracy_ranking_df.sort_values(['accuracy', 'macro_f1', 'recall'], ascending=False).head(30).reset_index(drop=True))
display(Markdown('## Fine threshold ranking by accuracy'))
display(fine_threshold_df.sort_values(['accuracy', 'macro_f1', 'recall'], ascending=False).head(30).reset_index(drop=True))


# 1–2. Ranking by accuracy + fine threshold search

## Coarse ranking by accuracy

,model,exp_name,aggregation,threshold,n,tn,fp,fn,tp,accuracy,precision,recall,f1-score,macro_f1,recall_rendah,recall_tinggi,recall_gap,balance_score
0,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,mean,0.200,692,141,181,108,262,0.582370,0.578844,0.572998,0.569198,0.569198,0.437888,0.708108,0.270220,0.298979
1,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,mean,0.150,692,96,226,63,307,0.582370,0.589879,0.563933,0.539562,0.539562,0.298137,0.829730,0.531593,0.007969
2,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,mean,0.175,692,118,204,88,282,0.578035,0.576531,0.564311,0.552924,0.552924,0.366460,0.762162,0.395703,0.157222
3,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,p90,0.550,692,205,117,177,193,0.575145,0.579615,0.579134,0.575017,0.575017,0.636646,0.521622,0.115024,0.459992
4,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,p90,0.525,692,191,131,166,204,0.570809,0.571985,0.572260,0.570658,0.570658,0.593168,0.551351,0.041816,0.528841
5,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,p90,0.500,692,178,144,153,217,0.570809,0.569436,0.569641,0.569442,0.569442,0.552795,0.586486,0.033691,0.535750
6,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,mean,0.225,692,163,159,138,232,0.570809,0.567439,0.566619,0.566499,0.566499,0.506211,0.627027,0.120816,0.445683
7,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,p90,0.375,692,118,204,93,277,0.570809,0.567563,0.557554,0.546888,0.546888,0.366460,0.748649,0.382189,0.164699
8,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,p90,0.225,692,55,267,30,340,0.570809,0.603595,0.544863,0.483139,0.483139,0.170807,0.918919,0.748111,-0.264972
9,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,p90,0.350,692,106,216,82,288,0.569364,0.567629,0.553785,0.537363,0.537363,0.329193,0.778378,0.449186,0.088177


## Fine threshold ranking by accuracy

,model,exp_name,aggregation,threshold,n,tn,fp,fn,tp,accuracy,precision,recall,f1-score,macro_f1,recall_rendah,recall_tinggi,recall_gap,balance_score
0,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,mean,0.155,692,100,222,66,304,0.583815,0.590178,0.566090,0.544204,0.544204,0.310559,0.821622,0.511063,0.033141
1,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,mean,0.200,692,141,181,108,262,0.582370,0.578844,0.572998,0.569198,0.569198,0.437888,0.708108,0.270220,0.298979
2,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,mean,0.195,692,137,185,104,266,0.582370,0.579133,0.572192,0.567334,0.567334,0.425466,0.718919,0.293453,0.273881
3,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,mean,0.150,692,96,226,63,307,0.582370,0.589879,0.563933,0.539562,0.539562,0.298137,0.829730,0.531593,0.007969
4,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,mean,0.135,692,86,236,53,317,0.582370,0.595971,0.561919,0.529996,0.529996,0.267081,0.856757,0.589676,-0.059680
5,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,mean,0.165,692,108,214,76,294,0.580925,0.582848,0.564999,0.548291,0.548291,0.335404,0.794595,0.459191,0.089100
6,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,mean,0.145,692,92,230,60,310,0.580925,0.589669,0.561776,0.534752,0.534752,0.285714,0.837838,0.552124,-0.017371
7,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,mean,0.205,692,145,177,114,256,0.579480,0.575535,0.571101,0.568374,0.568374,0.450311,0.691892,0.241581,0.326793
8,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,mean,0.190,692,133,189,102,268,0.579480,0.576195,0.568684,0.562842,0.562842,0.413043,0.724324,0.311281,0.251561
9,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,mean,0.160,692,102,220,71,299,0.579480,0.582852,0.562439,0.542394,0.542394,0.316770,0.808108,0.491338,0.051056


In [9]:
display(Markdown('# 3. Weighted ensemble'))

base_tables = {}
for spec in MODEL_SPECS:
    model_frames = frame_predictions_df[frame_predictions_df['model_label'] == spec['label']].copy()
    best = fine_threshold_df[fine_threshold_df['model'] == spec['label']].sort_values(['accuracy', 'macro_f1'], ascending=False).iloc[0]
    base_tables[spec['label']] = event_prob_table(model_frames, spec, str(best['aggregation']))

labels = [spec['label'] for spec in MODEL_SPECS]
base = base_tables[labels[0]][['event_key', 'kode_event', 'condition', 'label', 'true_label']].set_index('event_key')
prob_cols = []
for label in labels:
    prob_cols.append(base_tables[label].set_index('event_key')['agg_prob_anxiety_tinggi'].rename(label))
prob_df = pd.concat(prob_cols, axis=1)

ensemble_rows = []
ensemble_lookup = {}
for w0 in WEIGHT_GRID:
    for w1 in WEIGHT_GRID:
        w2 = round(1.0 - float(w0) - float(w1), 2)
        if w2 < 0 or w2 > 1:
            continue
        weights = np.array([w0, w1, w2], dtype=float)
        probs = (prob_df[labels].to_numpy() * weights).sum(axis=1)
        ens_base = base.copy()
        ens_base['agg_prob_anxiety_tinggi'] = probs
        ens_base = ens_base.reset_index()
        for threshold in THRESHOLDS:
            pred = apply_threshold(ens_base, float(threshold), 'weighted_ensemble', f'w={weights.tolist()}')
            row = metrics_for_name(pred, 'weighted_ensemble', f'w={weights.tolist()}', float(threshold))
            row.update({'w0': w0, 'w1': w1, 'w2': w2})
            ensemble_rows.append(row)
            ensemble_lookup[(w0, w1, w2, float(threshold))] = pred

weighted_ensemble_df = add_balance_cols(pd.DataFrame(ensemble_rows))
display(weighted_ensemble_df.sort_values(['accuracy', 'macro_f1', 'recall'], ascending=False).head(30).reset_index(drop=True))


# 3. Weighted ensemble

,model,exp_name,aggregation,threshold,n,tn,fp,fn,tp,accuracy,...,recall,f1-score,macro_f1,w0,w1,w2,recall_rendah,recall_tinggi,recall_gap,balance_score
0,weighted_ensemble,,"w=[0.1, 0.9, 0.0]",0.225,692,143,179,108,262,0.585260,...,0.576104,0.572622,0.572622,0.1,0.9,0.0,0.444099,0.708108,0.264009,0.308613
1,weighted_ensemble,,"w=[0.2, 0.8, 0.0]",0.200,692,99,223,65,305,0.583815,...,0.565889,0.543347,0.543347,0.2,0.8,0.0,0.307453,0.824324,0.516871,0.026476
2,weighted_ensemble,,"w=[0.0, 1.0, 0.0]",0.200,692,141,181,108,262,0.582370,...,0.572998,0.569198,0.569198,0.0,1.0,0.0,0.437888,0.708108,0.270220,0.298979
3,weighted_ensemble,,"w=[0.0, 1.0, 0.0]",0.150,692,96,226,63,307,0.582370,...,0.563933,0.539562,0.539562,0.0,1.0,0.0,0.298137,0.829730,0.531593,0.007969
4,weighted_ensemble,,"w=[0.2, 0.8, 0.0]",0.250,692,143,179,111,259,0.580925,...,0.572050,0.568808,0.568808,0.2,0.8,0.0,0.444099,0.700000,0.255901,0.312908
5,weighted_ensemble,,"w=[0.1, 0.8, 0.1]",0.250,692,139,183,108,262,0.579480,...,0.569893,0.565761,0.565761,0.1,0.8,0.1,0.431677,0.708108,0.276431,0.289330
6,weighted_ensemble,,"w=[0.0, 0.8, 0.2]",0.225,692,108,214,77,293,0.579480,...,0.563648,0.547111,0.547111,0.0,0.8,0.2,0.335404,0.791892,0.456488,0.090623
7,weighted_ensemble,,"w=[0.1, 0.9, 0.0]",0.150,692,75,247,44,326,0.579480,...,0.557000,0.515773,0.515773,0.1,0.9,0.0,0.232919,0.881081,0.648162,-0.132389
8,weighted_ensemble,,"w=[0.0, 1.0, 0.0]",0.175,692,118,204,88,282,0.578035,...,0.564311,0.552924,0.552924,0.0,1.0,0.0,0.366460,0.762162,0.395703,0.157222
9,weighted_ensemble,,"w=[0.1, 0.9, 0.0]",0.175,692,93,229,63,307,0.578035,...,0.559275,0.533413,0.533413,0.1,0.9,0.0,0.288820,0.829730,0.540910,-0.007497


In [10]:
display(Markdown('# 4. Fallback rule'))

best_mean_row = fine_threshold_df[
    (fine_threshold_df['model'] == 'qwalk_q12_q3_q4') & (fine_threshold_df['aggregation'] == 'mean')
].sort_values(['accuracy', 'macro_f1'], ascending=False).iloc[0]
best_p90_row = fine_threshold_df[
    (fine_threshold_df['model'] == 'qwalk_q12_q3_q4') & (fine_threshold_df['aggregation'] == 'p90')
].sort_values(['macro_f1', 'accuracy'], ascending=False).iloc[0]
qwalk_frames = frame_predictions_df[frame_predictions_df['model_label'] == 'qwalk_q12_q3_q4'].copy()
qwalk_spec = next(s for s in MODEL_SPECS if s['label'] == 'qwalk_q12_q3_q4')
mean_base = event_prob_table(qwalk_frames, qwalk_spec, 'mean').set_index('event_key')
p90_base = event_prob_table(qwalk_frames, qwalk_spec, 'p90').set_index('event_key')
fallback_rows = []
fallback_lookup = {}
for margin in np.round(np.arange(0.00, 0.151, 0.01), 3):
    pred = mean_base.copy()
    mean_prob = mean_base['agg_prob_anxiety_tinggi']
    p90_prob = p90_base['agg_prob_anxiety_tinggi']
    mean_thr = float(best_mean_row['threshold'])
    p90_thr = float(best_p90_row['threshold'])
    mean_pred = (mean_prob >= mean_thr).astype(int)
    p90_pred = (p90_prob >= p90_thr).astype(int)
    ambiguous = (mean_prob - mean_thr).abs() <= float(margin)
    pred['agg_prob_anxiety_tinggi'] = mean_prob
    pred['pred_label'] = mean_pred.where(~ambiguous, p90_pred).astype(int)
    pred['threshold'] = mean_thr
    pred['aggregation'] = f'mean_fallback_p90_margin_{margin:.3f}'
    pred['model_label'] = 'fallback_rule'
    pred = pred.reset_index()
    row = metrics_for_name(pred, 'fallback_rule', f'mean_fallback_p90_margin_{margin:.3f}', mean_thr)
    row['margin'] = margin
    fallback_rows.append(row)
    fallback_lookup[float(margin)] = pred
fallback_df = add_balance_cols(pd.DataFrame(fallback_rows))
display(fallback_df.sort_values(['accuracy', 'macro_f1'], ascending=False).reset_index(drop=True))


# 4. Fallback rule

,model,exp_name,aggregation,threshold,n,tn,fp,fn,tp,accuracy,precision,recall,f1-score,macro_f1,margin,recall_rendah,recall_tinggi,recall_gap,balance_score
0,fallback_rule,,mean_fallback_p90_margin_0.000,0.155,692,100,222,66,304,0.583815,0.590178,0.566090,0.544204,0.544204,0.00,0.310559,0.821622,0.511063,0.033141
1,fallback_rule,,mean_fallback_p90_margin_0.020,0.155,692,88,234,55,315,0.582370,0.594578,0.562322,0.532011,0.532011,0.02,0.273292,0.851351,0.578059,-0.046048
2,fallback_rule,,mean_fallback_p90_margin_0.010,0.155,692,92,230,60,310,0.580925,0.589669,0.561776,0.534752,0.534752,0.01,0.285714,0.837838,0.552124,-0.017371
3,fallback_rule,,mean_fallback_p90_margin_0.070,0.155,692,59,263,32,338,0.573699,0.605374,0.548372,0.490952,0.490952,0.07,0.183230,0.913514,0.730284,-0.239332
4,fallback_rule,,mean_fallback_p90_margin_0.080,0.155,692,59,263,32,338,0.573699,0.605374,0.548372,0.490952,0.490952,0.08,0.183230,0.913514,0.730284,-0.239332
5,fallback_rule,,mean_fallback_p90_margin_0.090,0.155,692,59,263,32,338,0.573699,0.605374,0.548372,0.490952,0.490952,0.09,0.183230,0.913514,0.730284,-0.239332
6,fallback_rule,,mean_fallback_p90_margin_0.100,0.155,692,59,263,32,338,0.573699,0.605374,0.548372,0.490952,0.490952,0.10,0.183230,0.913514,0.730284,-0.239332
7,fallback_rule,,mean_fallback_p90_margin_0.110,0.155,692,59,263,32,338,0.573699,0.605374,0.548372,0.490952,0.490952,0.11,0.183230,0.913514,0.730284,-0.239332
8,fallback_rule,,mean_fallback_p90_margin_0.120,0.155,692,59,263,32,338,0.573699,0.605374,0.548372,0.490952,0.490952,0.12,0.183230,0.913514,0.730284,-0.239332
9,fallback_rule,,mean_fallback_p90_margin_0.130,0.155,692,59,263,32,338,0.573699,0.605374,0.548372,0.490952,0.490952,0.13,0.183230,0.913514,0.730284,-0.239332


In [11]:
display(Markdown('# 5. Threshold per question'))

question_rows = []
question_predictions = []
for spec in MODEL_SPECS:
    model_frames = frame_predictions_df[frame_predictions_df['model_label'] == spec['label']].copy()
    for method in EVENT_METHODS:
        base = event_prob_table(model_frames, spec, method)
        parts = []
        thresholds = {}
        for question, g in base.groupby('question', sort=False):
            local_rows = []
            local_preds = {}
            for threshold in THRESHOLDS:
                pred = apply_threshold(g, float(threshold), spec['label'], method)
                local_rows.append(metrics_dict(pred, spec, method, float(threshold)))
                local_preds[float(threshold)] = pred
            local_df = pd.DataFrame(local_rows)
            best = local_df.sort_values(['accuracy', 'macro_f1'], ascending=False).iloc[0]
            thresholds[question] = float(best['threshold'])
            parts.append(local_preds[float(best['threshold'])])
        pred_all = pd.concat(parts, ignore_index=True)
        row = metrics_for_name(pred_all, f'{spec["label"]}_per_question', method, -1)
        row['thresholds'] = json.dumps(thresholds, sort_keys=True)
        question_rows.append(row)
        question_predictions.append(pred_all.assign(strategy='per_question'))
per_question_df = add_balance_cols(pd.DataFrame(question_rows))
display(per_question_df.sort_values(['accuracy', 'macro_f1'], ascending=False).head(30).reset_index(drop=True))


# 5. Threshold per question

,model,exp_name,aggregation,threshold,n,tn,fp,fn,tp,accuracy,precision,recall,f1-score,macro_f1,thresholds,recall_rendah,recall_tinggi,recall_gap,balance_score
0,qwalk_q12_q3_q4_per_question,,mean,-1.0,692,165,157,124,246,0.593931,0.590678,0.588644,0.588290,0.588290,"{""q1"": 0.225, ""q2"": 0.2, ""q3"": 0.15, ""q4"": 0.3...",0.512422,0.664865,0.152443,0.435847
1,qwalk_q12_q3_q4_per_question,,p90,-1.0,692,141,181,100,270,0.593931,0.591866,0.583809,0.579311,0.579311,"{""q1"": 0.275, ""q2"": 0.3, ""q3"": 0.45, ""q4"": 0.5...",0.437888,0.729730,0.291842,0.287470
2,qwalk_q12_q3_q4_per_question,,top3_mean,-1.0,692,150,172,115,255,0.585260,0.581614,0.577514,0.575486,0.575486,"{""q1"": 0.525, ""q2"": 0.25, ""q3"": 0.375, ""q4"": 0...",0.465839,0.689189,0.223351,0.352136
3,qholdout_q123_q4_q5_per_question,,p90,-1.0,692,158,164,124,246,0.583815,0.580142,0.577774,0.576974,0.576974,"{""q1"": 0.65, ""q2"": 0.45, ""q3"": 0.225, ""q4"": 0....",0.490683,0.664865,0.174182,0.402792
4,qholdout_q123_q4_q5_per_question,,max,-1.0,692,145,177,112,258,0.582370,0.578653,0.573804,0.570929,0.570929,"{""q1"": 0.75, ""q2"": 0.675, ""q3"": 0.275, ""q4"": 0...",0.450311,0.697297,0.246987,0.323942
5,qholdout_q123_q4_q5_per_question,,top3_mean,-1.0,692,146,176,119,251,0.573699,0.569383,0.565897,0.563653,0.563653,"{""q1"": 0.625, ""q2"": 0.45, ""q3"": 0.175, ""q4"": 0...",0.453416,0.678378,0.224962,0.338691
6,qwalk_q12_q3_q4_per_question,,max,-1.0,692,111,211,85,285,0.572254,0.570462,0.557495,0.543385,0.543385,"{""q1"": 0.725, ""q2"": 0.35, ""q3"": 0.55, ""q4"": 0....",0.344720,0.770270,0.425550,0.117835
7,qholdout_q123_q4_q5_per_question,,mean,-1.0,692,141,181,118,252,0.567919,0.563194,0.559485,0.556508,0.556508,"{""q1"": 0.4, ""q2"": 0.2, ""q3"": 0.1, ""q4"": 0.225,...",0.437888,0.681081,0.243193,0.313315
8,qwalk_q123_q4_q5_per_question,,p90,-1.0,692,85,237,65,305,0.563584,0.564699,0.544150,0.514515,0.514515,"{""q1"": 0.8, ""q2"": 0.45, ""q3"": 0.6, ""q4"": 0.85,...",0.263975,0.824324,0.560349,-0.045835
9,qwalk_q123_q4_q5_per_question,,mean,-1.0,692,72,250,53,317,0.562139,0.567541,0.540180,0.499388,0.499388,"{""q1"": 0.525, ""q2"": 0.05, ""q3"": 0.3, ""q4"": 0.3...",0.223602,0.856757,0.633154,-0.133767


In [12]:
display(Markdown('# 6. Threshold per date'))

date_rows = []
date_predictions = []
for spec in MODEL_SPECS:
    model_frames = frame_predictions_df[frame_predictions_df['model_label'] == spec['label']].copy()
    for method in EVENT_METHODS:
        base = event_prob_table(model_frames, spec, method)
        parts = []
        thresholds = {}
        for date, g in base.groupby(base['event_key'].str.split('|').str[0], sort=False):
            local_rows = []
            local_preds = {}
            for threshold in THRESHOLDS:
                pred = apply_threshold(g, float(threshold), spec['label'], method)
                local_rows.append(metrics_dict(pred, spec, method, float(threshold)))
                local_preds[float(threshold)] = pred
            local_df = pd.DataFrame(local_rows)
            best = local_df.sort_values(['accuracy', 'macro_f1'], ascending=False).iloc[0]
            thresholds[date] = float(best['threshold'])
            parts.append(local_preds[float(best['threshold'])])
        pred_all = pd.concat(parts, ignore_index=True)
        row = metrics_for_name(pred_all, f'{spec["label"]}_per_date', method, -1)
        row['thresholds'] = json.dumps(thresholds, sort_keys=True)
        date_rows.append(row)
        date_predictions.append(pred_all.assign(strategy='per_date'))
per_date_df = add_balance_cols(pd.DataFrame(date_rows))
display(per_date_df.sort_values(['accuracy', 'macro_f1'], ascending=False).head(30).reset_index(drop=True))


# 6. Threshold per date

,model,exp_name,aggregation,threshold,n,tn,fp,fn,tp,accuracy,precision,recall,f1-score,macro_f1,thresholds,recall_rendah,recall_tinggi,recall_gap,balance_score
0,qwalk_q12_q3_q4_per_date,,mean,-1.0,692,154,168,105,265,0.605491,0.603302,0.597239,0.595073,0.595073,"{""15_04_2026"": 0.15, ""17_04_2026"": 0.35}",0.478261,0.716216,0.237955,0.357117
1,qwalk_q12_q3_q4_per_date,,p90,-1.0,692,100,222,61,309,0.591040,0.601519,0.572847,0.549992,0.549992,"{""15_04_2026"": 0.225, ""17_04_2026"": 0.55}",0.310559,0.835135,0.524576,0.025415
2,qwalk_q12_q3_q4_per_date,,max,-1.0,692,104,218,77,293,0.573699,0.573986,0.557437,0.539336,0.539336,"{""15_04_2026"": 0.35, ""17_04_2026"": 0.775}",0.322981,0.791892,0.468911,0.070426
3,qwalk_q12_q3_q4_per_date,,top3_mean,-1.0,692,130,192,105,265,0.570809,0.566530,0.559971,0.553828,0.553828,"{""15_04_2026"": 0.375, ""17_04_2026"": 0.525}",0.403727,0.716216,0.312490,0.241339
4,qwalk_q123_q4_q5_per_date,,mean,-1.0,692,146,176,130,240,0.557803,0.552954,0.551032,0.549491,0.549491,"{""15_04_2026"": 0.325, ""17_04_2026"": 0.85}",0.453416,0.648649,0.195232,0.354258
5,qholdout_q123_q4_q5_per_date,,p90,-1.0,692,280,42,264,106,0.557803,0.615461,0.578026,0.527959,0.527959,"{""15_04_2026"": 0.65, ""17_04_2026"": 0.925}",0.869565,0.286486,0.583079,-0.055120
6,qholdout_q123_q4_q5_per_date,,mean,-1.0,692,143,179,130,240,0.553468,0.548301,0.546374,0.544519,0.544519,"{""15_04_2026"": 0.1, ""17_04_2026"": 0.675}",0.444099,0.648649,0.204549,0.339969
7,qwalk_q123_q4_q5_per_date,,p90,-1.0,692,32,290,21,349,0.550578,0.574970,0.521311,0.431220,0.431220,"{""15_04_2026"": 0.5, ""17_04_2026"": 0.575}",0.099379,0.943243,0.843864,-0.412644
8,qholdout_q123_q4_q5_per_date,,top3_mean,-1.0,692,273,49,263,107,0.549133,0.597613,0.568508,0.521604,0.521604,"{""15_04_2026"": 0.625, ""17_04_2026"": 0.95}",0.847826,0.289189,0.558637,-0.037033
9,qholdout_q123_q4_q5_per_date,,max,-1.0,692,249,73,243,127,0.543353,0.570549,0.558268,0.528704,0.528704,"{""15_04_2026"": 0.775, ""17_04_2026"": 0.95}",0.773292,0.343243,0.430049,0.098655


In [13]:
display(Markdown('# 7. Event-noise filtering by frame count'))

filter_rows = []
filter_predictions = []
frame_ranges = [(1, 999), (2, 999), (3, 999), (1, 20), (2, 20), (3, 20), (1, 15), (2, 15), (3, 15), (1, 10), (2, 10), (3, 10), (1, 5), (2, 5), (3, 5)]
for spec in MODEL_SPECS:
    model_frames = frame_predictions_df[frame_predictions_df['model_label'] == spec['label']].copy()
    for method in EVENT_METHODS:
        base = event_prob_table(model_frames, spec, method)
        for min_frames, max_frames in frame_ranges:
            filt = base[(base['n_frames'] >= min_frames) & (base['n_frames'] <= max_frames)].copy()
            if filt['true_label'].nunique() < 2 or len(filt) < 20:
                continue
            local_rows = []
            local_preds = {}
            for threshold in THRESHOLDS:
                pred = apply_threshold(filt, float(threshold), spec['label'], method)
                local_rows.append(metrics_dict(pred, spec, method, float(threshold)))
                local_preds[float(threshold)] = pred
            local_df = pd.DataFrame(local_rows)
            best = local_df.sort_values(['accuracy', 'macro_f1'], ascending=False).iloc[0]
            row = best.to_dict()
            row.update({'min_frames': min_frames, 'max_frames': max_frames, 'coverage': len(filt) / len(base)})
            filter_rows.append(row)
            filter_predictions.append(local_preds[float(best['threshold'])].assign(strategy='frame_filter', min_frames=min_frames, max_frames=max_frames))
filter_df = add_balance_cols(pd.DataFrame(filter_rows))
display(filter_df.sort_values(['accuracy', 'macro_f1'], ascending=False).head(40).reset_index(drop=True))


# 7. Event-noise filtering by frame count

,model,exp_name,aggregation,threshold,n,tn,fp,fn,tp,accuracy,...,recall,f1-score,macro_f1,min_frames,max_frames,coverage,recall_rendah,recall_tinggi,recall_gap,balance_score
0,qholdout_q123_q4_q5,convat_apex_anxiety_qholdout_q123_q4_q5,p90,0.050,78,0,16,0,62,0.794872,...,0.500000,0.442857,0.442857,1,5,0.112717,0.000000,1.000000,1.000000,-0.557143
1,qholdout_q123_q4_q5,convat_apex_anxiety_qholdout_q123_q4_q5,p90,0.050,78,0,16,0,62,0.794872,...,0.500000,0.442857,0.442857,2,5,0.112717,0.000000,1.000000,1.000000,-0.557143
2,qholdout_q123_q4_q5,convat_apex_anxiety_qholdout_q123_q4_q5,p90,0.050,78,0,16,0,62,0.794872,...,0.500000,0.442857,0.442857,3,5,0.112717,0.000000,1.000000,1.000000,-0.557143
3,qholdout_q123_q4_q5,convat_apex_anxiety_qholdout_q123_q4_q5,max,0.050,78,0,16,0,62,0.794872,...,0.500000,0.442857,0.442857,1,5,0.112717,0.000000,1.000000,1.000000,-0.557143
4,qholdout_q123_q4_q5,convat_apex_anxiety_qholdout_q123_q4_q5,max,0.050,78,0,16,0,62,0.794872,...,0.500000,0.442857,0.442857,2,5,0.112717,0.000000,1.000000,1.000000,-0.557143
5,qholdout_q123_q4_q5,convat_apex_anxiety_qholdout_q123_q4_q5,max,0.050,78,0,16,0,62,0.794872,...,0.500000,0.442857,0.442857,3,5,0.112717,0.000000,1.000000,1.000000,-0.557143
6,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,top3_mean,0.050,78,0,16,0,62,0.794872,...,0.500000,0.442857,0.442857,1,5,0.112717,0.000000,1.000000,1.000000,-0.557143
7,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,top3_mean,0.050,78,0,16,0,62,0.794872,...,0.500000,0.442857,0.442857,2,5,0.112717,0.000000,1.000000,1.000000,-0.557143
8,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,top3_mean,0.050,78,0,16,0,62,0.794872,...,0.500000,0.442857,0.442857,3,5,0.112717,0.000000,1.000000,1.000000,-0.557143
9,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,p90,0.050,78,0,16,0,62,0.794872,...,0.500000,0.442857,0.442857,1,5,0.112717,0.000000,1.000000,1.000000,-0.557143


In [14]:
display(Markdown('# Final comparison'))
summary_parts = [
    fine_threshold_df.assign(strategy='fine_threshold'),
    weighted_ensemble_df.assign(strategy='weighted_ensemble'),
    fallback_df.assign(strategy='fallback_rule'),
    per_question_df.assign(strategy='per_question'),
    per_date_df.assign(strategy='per_date'),
    filter_df.assign(strategy='event_filter'),
]
experiment_summary_df = pd.concat(summary_parts, ignore_index=True, sort=False)
display(experiment_summary_df.sort_values(['accuracy', 'macro_f1', 'recall'], ascending=False).head(50).reset_index(drop=True))

with pd.ExcelWriter(EVAL_ROOT / 'accuracy_improvement_experiments.xlsx') as writer:
    datatest_table.to_excel(writer, sheet_name='tabel_datatest_event', index=False)
    accuracy_ranking_df.to_excel(writer, sheet_name='coarse_accuracy', index=False)
    fine_threshold_df.to_excel(writer, sheet_name='fine_threshold', index=False)
    weighted_ensemble_df.to_excel(writer, sheet_name='weighted_ensemble', index=False)
    fallback_df.to_excel(writer, sheet_name='fallback_rule', index=False)
    per_question_df.to_excel(writer, sheet_name='per_question', index=False)
    per_date_df.to_excel(writer, sheet_name='per_date', index=False)
    filter_df.to_excel(writer, sheet_name='event_filter', index=False)
    experiment_summary_df.to_excel(writer, sheet_name='final_comparison', index=False)
print(f'Saved: {EVAL_ROOT / "accuracy_improvement_experiments.xlsx"}')


# Final comparison

,model,exp_name,aggregation,threshold,n,tn,fp,fn,tp,accuracy,...,balance_score,strategy,w0,w1,w2,margin,thresholds,min_frames,max_frames,coverage
0,qholdout_q123_q4_q5,convat_apex_anxiety_qholdout_q123_q4_q5,p90,0.050,78,0,16,0,62,0.794872,...,-0.557143,event_filter,NaN,NaN,NaN,NaN,NaN,1.0,5.0,0.112717
1,qholdout_q123_q4_q5,convat_apex_anxiety_qholdout_q123_q4_q5,p90,0.050,78,0,16,0,62,0.794872,...,-0.557143,event_filter,NaN,NaN,NaN,NaN,NaN,2.0,5.0,0.112717
2,qholdout_q123_q4_q5,convat_apex_anxiety_qholdout_q123_q4_q5,p90,0.050,78,0,16,0,62,0.794872,...,-0.557143,event_filter,NaN,NaN,NaN,NaN,NaN,3.0,5.0,0.112717
3,qholdout_q123_q4_q5,convat_apex_anxiety_qholdout_q123_q4_q5,max,0.050,78,0,16,0,62,0.794872,...,-0.557143,event_filter,NaN,NaN,NaN,NaN,NaN,1.0,5.0,0.112717
4,qholdout_q123_q4_q5,convat_apex_anxiety_qholdout_q123_q4_q5,max,0.050,78,0,16,0,62,0.794872,...,-0.557143,event_filter,NaN,NaN,NaN,NaN,NaN,2.0,5.0,0.112717
5,qholdout_q123_q4_q5,convat_apex_anxiety_qholdout_q123_q4_q5,max,0.050,78,0,16,0,62,0.794872,...,-0.557143,event_filter,NaN,NaN,NaN,NaN,NaN,3.0,5.0,0.112717
6,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,top3_mean,0.050,78,0,16,0,62,0.794872,...,-0.557143,event_filter,NaN,NaN,NaN,NaN,NaN,1.0,5.0,0.112717
7,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,top3_mean,0.050,78,0,16,0,62,0.794872,...,-0.557143,event_filter,NaN,NaN,NaN,NaN,NaN,2.0,5.0,0.112717
8,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,top3_mean,0.050,78,0,16,0,62,0.794872,...,-0.557143,event_filter,NaN,NaN,NaN,NaN,NaN,3.0,5.0,0.112717
9,qwalk_q12_q3_q4,convat_apex_anxiety_qwalk_q12_q3_q4,p90,0.050,78,0,16,0,62,0.794872,...,-0.557143,event_filter,NaN,NaN,NaN,NaN,NaN,1.0,5.0,0.112717


Saved: /home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/output/apex/datatest/event_level_accuracy_experiments/accuracy_improvement_experiments.xlsx
